![image.png](attachment:cb2ec80a-6022-4709-abee-dcee2aa59ea0.png)

#### 1. Importar Bibliotecas

In [62]:
import sqlite3
import pandas as pd

#### 2. Criar conexão com banco de dados em memória

In [63]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

#### 3. Criar tabelas

In [64]:
# Tabela 1: Funcionarios
cursor.execute("""
CREATE TABLE Employees (
id INTEGER PRIMARY KEY,
name TEXT,
dep_id INTEGER
);
""")

# Tabela 2: Departamentos
cursor.execute("""
CREATE TABLE Departments (
id INTEGER PRIMARY KEY,
department TEXT
);
""")

#### 4. Inserir dados nas tabelas

In [65]:
# Inserir dados em Funcionarios
cursor.executemany("INSERT INTO Employees VALUES (?, ?, ?)", [
(1, 'Ana', 10),
(2, 'Bruno', 20),
(3, 'Carla', None),
(4, 'Daniel', 30)
])

# Inserir dados em Departamentos
cursor.executemany("INSERT INTO Departments VALUES (?, ?)", [
(10, 'RH'),
(20, 'TI'),
(40, 'Marketing')
])

conn.commit()

#### 5. Visualizar tabelas

In [66]:
cursor.execute("SELECT * FROM employees;")
pd.DataFrame(cursor.fetchall(), columns = ['id','name','dep_id'])

,id,name,dep_id
0,1,Ana,10.0
1,2,Bruno,20.0
2,3,Carla,NaN
3,4,Daniel,30.0


In [67]:
cursor.execute("SELECT * FROM departments;")
pd.DataFrame(cursor.fetchall(), columns = ['id','departament'])

,id,departament
0,10,RH
1,20,TI
2,40,Marketing


### INNER JOIN:

`INNER JOIN` é um formato comum de join, que retorna dados apenas quando as duas tabelas tem chaves correspondentes na cláusula ON do join.

![image.png](attachment:e884d3d3-bda8-4d25-a6cc-a4bad7842fa5.png)

In [68]:
cursor.execute('''
SELECT f.name, d.department
FROM Employees f
INNER JOIN Departments d
ON f.dep_id = d.id;
''')
inner_join_result = cursor.fetchall()
pd.DataFrame(inner_join_result, columns=["name", "department"])

,name,department
0,Ana,RH
1,Bruno,TI


### LEFT JOIN

`LEFT OUTER JOIN` ou simplesmente `LEFT JOIN` é um dos formatos mais usados de join, que retorna a Tabela A inteira e apenas os registros que coincidirem com a igualdade do join na Tabela B (ou campos nulos para os campos sem correspondência).

![image.png](attachment:d1ae10de-3f4e-45ae-8801-c559dae93c93.png)

In [69]:
cursor.execute('''
SELECT f.name, d.department
FROM Employees f
LEFT JOIN Departments d
ON f.dep_id = d.id;
''')
left_join_result = cursor.fetchall()
pd.DataFrame(left_join_result, columns=["name", "department"])

,name,department
0,Ana,RH
1,Bruno,TI
2,Carla,None
3,Daniel,None


### RIGHT JOIN

`RIGHT OUTER JOIN` ou simplesmente `RIGHT JOIN` segue o mesmo raciocínio do LEFT JOIN, mas se aplicando à tabela B em vez da A:

![image.png](attachment:c2686ae2-6aba-453d-9e1a-41d7819c005c.png)

In [70]:
cursor.execute('''
SELECT f.name, d.department
FROM Departments d
LEFT JOIN Employees f
ON f.dep_id = d.id;
''')
right_join_result = cursor.fetchall()
pd.DataFrame(right_join_result, columns=["name", "department"])

,name,department
0,Ana,RH
1,Bruno,TI
2,None,Marketing


### FULL JOIN

Conhecida como `OUTER JOIN` ou simplesmente `FULL JOIN`, este retorna todos os registros de ambas as tabelas.

![image.png](attachment:31aaf99a-4794-444e-b91a-11b3b009f758.png)

Obs: SQLite não suporta FULL JOIN diretamente. Então simulamos com UNION de LEFT JOIN e RIGHT JOIN.

In [71]:
cursor.execute('''
SELECT f.name, d.department
FROM Employees f
LEFT JOIN Departments d ON f.dep_id = d.id
UNION
SELECT f.name, d.department
FROM Departments d
LEFT JOIN Employees f ON f.dep_id = d.id;
''')
full_join_result = cursor.fetchall()
pd.DataFrame(full_join_result, columns=["name", "department"])

,name,department
0,None,Marketing
1,Ana,RH
2,Bruno,TI
3,Carla,None
4,Daniel,None


### CROSS JOIN: retorna o produto cartesiano entre as duas tabelas.

Basicamente é o produto cartesiano entre as duas tabelas. Para cada linha de Tabela A, são retornadas todas as linhas de Tabela B.

É mais fácil entender o CROSS JOIN como um "Join sem cláusula ON", ou seja, todas as combinações de linhas de A e B são devolvidas. Inclusive, se você fizer um CROSS JOIN com cláusula ON, ele "vira" um mero INNER JOIN.

![image.png](attachment:811c39d0-7b81-4cbc-9906-df13ef2d9cbf.png)

In [72]:
cursor.execute('''
SELECT f.name, d.department
FROM Employees f
CROSS JOIN Departments d;
''')
cross_join_result = cursor.fetchall()
pd.DataFrame(cross_join_result, columns=["name", "department"])

,name,department
0,Ana,RH
1,Ana,TI
2,Ana,Marketing
3,Bruno,RH
4,Bruno,TI
5,Bruno,Marketing
6,Carla,RH
7,Carla,TI
8,Carla,Marketing
9,Daniel,RH


In [74]:
cursor.execute('''
SELECT f.name, d.department
FROM Employees f
FULL OUTER JOIN Departments d;
''')
cross_join_result = cursor.fetchall()
pd.DataFrame(cross_join_result, columns=["name", "department"])

,name,department
0,Ana,RH
1,Ana,TI
2,Ana,Marketing
3,Bruno,RH
4,Bruno,TI
5,Bruno,Marketing
6,Carla,RH
7,Carla,TI
8,Carla,Marketing
9,Daniel,RH
